In [1]:
import xpress as xp
import pandas as pd
import numpy as np
from datetime import datetime

class TimetablingModel:
    """University timetabling model using Xpress solver with fast extraction."""
    
    def __init__(self, students_df, events_df, weeks_df, rooms_df):
        self.students_df = students_df
        self.events_df = events_df
        self.weeks_df = weeks_df
        self.rooms_df = rooms_df
        self.model = xp.problem(name="Timetabling")
        
        # Dimensions
        self.events, self.weeks, self.rooms = [], [], []
        self.days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
        self.time_slots = [f"{h:02d}:{m:02d}" for h in range(9, 18) for m in [0, 30]]
        
        # Parameters
        self.event_size, self.event_duration, self.event_weeks = {}, {}, {}
        self.event_name, self.room_capacity, self.room_campus = {}, {}, {}
        self.room_building, self.curricula = {}, {}
        
        # Decision variables
        self.x, self.v = {}, {}
        print("✓ Model initialized")

    def build_data(self, max_events=50, max_rooms=20):
        print("\nBuilding data structures...")
        
        # 1. Weeks (All teaching weeks)
        teaching_weeks = self.weeks_df[self.weeks_df['Week Type'] == 'Other']['Week Number'].unique()
        if len(teaching_weeks) == 0: teaching_weeks = self.weeks_df['Week Number'].unique()
        self.weeks = sorted([int(w) for w in teaching_weeks if pd.notna(w)])
        print(f"  Weeks: {len(self.weeks)} ({self.weeks})")

        # 2. Events
        self.events = self.events_df['Event ID'].unique()[:max_events].tolist()
        for e in self.events:
            row = self.events_df[self.events_df['Event ID'] == e].iloc[0]
            self.event_size[e] = int(row.get('Event Size', 0)) if pd.notna(row.get('Event Size')) else 0
            self.event_name[e] = row.get('Event Name', 'Unknown')
            dur = row.get('Duration (minutes)', 60)
            self.event_duration[e] = max(1, int(np.ceil(float(dur) / 30)))
            self.event_weeks[e] = self.weeks # Assign to all weeks found

        # 3. Rooms (Exclude Holyrood)
        room_count = 0
        for _, row in self.rooms_df.iterrows():
            if room_count >= max_rooms: break
            if row.get('Campus') == 'Holyrood': continue
            r_id = row['Id']
            self.rooms.append(r_id)
            self.room_capacity[r_id] = int(row.get('Capacity', 0))
            self.room_campus[r_id] = row.get('Campus', 'Central')
            self.room_building[r_id] = row.get('Building_Name', 'Unknown')
            room_count += 1
        print(f"  Rooms: {len(self.rooms)}")

        # 4. Curricula
        student_events = self.students_df.groupby('AnonID')['Event ID'].apply(set).to_dict()
        c_id = 0
        for events in list(student_events.values())[:5000]:
            relevant = [e for e in events if e in self.events]
            if len(relevant) > 1:
                self.curricula[c_id] = relevant
                c_id += 1
        print(f"  Curricula: {len(self.curricula)}\n✓ Data structures built")

    def build_model(self):
        print("Building MIP model...")
        # Create Variables
        for e in self.events:
            for w in self.event_weeks.get(e, self.weeks[:1]):
                for d in self.days:
                    for t in self.time_slots:
                        for r in self.rooms:
                            self.x[e,d,t,r,w] = xp.var(vartype=xp.binary)
                            self.v[e,d,t,r,w] = xp.var(lb=0)
        self.model.addVariable(list(self.x.values()) + list(self.v.values()))

        # Objective
        obj = []
        for v_var in self.v.values(): obj.append(100 * v_var)
        for (e,d,t,r,w), x_var in self.x.items():
            camp = self.room_campus.get(r, 'Central')
            pen = 10 if camp == 'Lauriston' else (20 if camp == 'New College' else 0)
            if pen > 0: obj.append(pen * x_var)
        self.model.setObjective(xp.Sum(obj), sense=xp.minimize)

        # Constraints
        # 1. Weekly Completeness
        for e in self.events:
            for w in self.weeks:
                vars_ev = [self.x[e,d,t,r,w] for d in self.days for t in self.time_slots for r in self.rooms if (e,d,t,r,w) in self.x]
                if vars_ev: self.model.addConstraint(xp.Sum(vars_ev) == 1)

        # 2. Duration-Aware Room Uniqueness
        for r in self.rooms:
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        occupants = []
                        for e in self.events:
                            dur = self.event_duration[e]
                            for lookback in range(dur):
                                if t_idx - lookback >= 0:
                                    t_p = self.time_slots[t_idx - lookback]
                                    if (e,d,t_p,r,w) in self.x: occupants.append(self.x[e,d,t_p,r,w])
                        if occupants: self.model.addConstraint(xp.Sum(occupants) <= 1)

        # 3. Duration-Aware Curriculum Clash
        for events in self.curricula.values():
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        clash_vars = []
                        for e in events:
                            dur = self.event_duration[e]
                            for lookback in range(dur):
                                if t_idx - lookback >= 0:
                                    t_p = self.time_slots[t_idx - lookback]
                                    for r in self.rooms:
                                        if (e,d,t_p,r,w) in self.x: clash_vars.append(self.x[e,d,t_p,r,w])
                        if clash_vars: self.model.addConstraint(xp.Sum(clash_vars) <= 1)

        # 4. Capacity Logic
        for (e,d,t,r,w), x_v in self.x.items():
            sz, cp, v_v = self.event_size[e], self.room_capacity[r], self.v[e,d,t,r,w]
            self.model.addConstraint(v_v >= (0.5 * cp - sz) * x_v) # Underfill
            self.model.addConstraint(v_v >= (sz - cp) * x_v)      # Overfill
        print(f"✓ Model built: {self.model.attributes.cols} vars")

    def solve(self, time_limit=300):
        print(f"Solving (Limit: {time_limit}s)...")
        self.model.controls.maxtime = -time_limit
        self.model.controls.miprelstop = 0.05
        self.model.solve()
        return self.model.attributes.solstatus in [xp.SolStatus.FEASIBLE, xp.SolStatus.OPTIMAL]

    def extract_solution(self):
        """Vectorised extraction - skips the 30min wait."""
        print("🚀 Starting Lightning Extraction...")
        vars_list = list(self.x.values())
        keys_list = list(self.x.keys())
        sol_values = self.model.getSolution(vars_list)
        
        results = [
            {'Event_ID': k[0], 'Week': k[4], 'Day': k[1], 'Time': k[2], 'Room_ID': k[3], 
             'Event_Name': self.event_name.get(k[0]), 'Campus': self.room_campus.get(k[3]),
             'Event_Size': self.event_size.get(k[0]), 'Room_Capacity': self.room_capacity.get(k[3])}
            for i, k in enumerate(keys_list) if sol_values[i] > 0.5
        ]
        df = pd.DataFrame(results)
        if not df.empty:
            day_map = {d: i for i, d in enumerate(self.days)}
            df['d_idx'] = df['Day'].map(day_map)
            df = df.sort_values(['Week', 'd_idx', 'Time']).drop('d_idx', axis=1)
        return df

def run_optimization():
    # Load Data
    try:
        e_df, r_df = pd.read_csv('Event_data.csv'), pd.read_csv('Rooms_data.csv')
        s_df, st_df = pd.read_csv('Semester1.csv'), pd.read_csv('Student_data.csv')
    except Exception as err:
        print(f"File Error: {err}"); return

    # Execute
    model = TimetablingModel(st_df, e_df, s_df, r_df)
    model.build_data(max_events=30, max_rooms=10) # Testing settings
    model.build_model()
    
    if model.solve():
        df = model.extract_solution()
        df.to_csv('final_schedule.csv', index=False)
        print(f"\n✅ SUCCESS! Saved {len(df)} events to final_schedule.csv")
        
        # Quick Stats
        over = (df['Event_Size'] > df['Room_Capacity']).sum()
        print(f"Stats: {over} events exceed room capacity.")
    else:
        print("❌ No solution found.")

if __name__ == "__main__":
    run_optimization()


C:\Users\Selina Kanguha\AppData\Local\Temp\ipykernel_8568\2991890951.py:14: LicenseWarning: Using the license file found in your Xpress installation. If you want to use this license and no longer want to see this message, use the following code before using the xpress module:
  xpress.init('C:/xpressmp//bin/xpauth.xpr')
  self.model = xp.problem(name="Timetabling")


✓ Model initialized

Building data structures...
  Weeks: 11 ([9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
  Rooms: 10
  Curricula: 1
✓ Data structures built
Building MIP model...


C:\Users\Selina Kanguha\AppData\Local\Temp\ipykernel_8568\2991890951.py:80: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  self.x[e,d,t,r,w] = xp.var(vartype=xp.binary)
C:\Users\Selina Kanguha\AppData\Local\Temp\ipykernel_8568\2991890951.py:81: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  self.v[e,d,t,r,w] = xp.var(lb=0)


✓ Model built: 594000 vars
Solving (Limit: 300s)...
FICO Xpress v9.7.0, Hyper, solve started 19:43:58, Mar 11, 2026
Heap usage: 345MB (peak 345MB, 4785KB system)
Minimizing MILP Timetabling using up to 22 threads and up to 15GB memory, with these control settings:
MAXTIME = -300
OUTPUTLOG = 1
MIPRELSTOP = .05
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
    605220 rows       594000 cols      2280850 elements    297000 entities
Presolved problem has:
      9790 rows       269500 cols       962500 elements    269500 entities
Presolve finished in 9 seconds
Heap usage: 555MB (peak 907MB, 4785KB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 1.00e+00,  1.92e+02] / [ 1.00e+00,  1.00e+00]
  RHS and bounds [min,max] : [ 1.00e+00,  1.00e+00] / [ 1.00e+00,  1.00e+00]
  Objective      [min,max] : [ 1.00e+02,  1.00e+02] / [ 1.00e+02,  1.84e+04]
Autoscaling applied standard scaling

Will try t